# Synthetic GMM Data Sanity Notebook

This notebook is only about the data. Before we train anything, we want to see that the canonical `gmm_subspace` generator is making the objects we think it is making.

We will check three cases:

1. a real 2D GMM where `d = D = 2`
2. a projected high-dimensional GMM where `d = 2` and `D` is large
3. a fixed-ratio sweep where `d / D` is fixed while `d` increases

The checks are deliberately direct: print tensor shapes, inspect labels, verify the subspace basis, project back to latent coordinates, run PCA/t-SNE/UMAP, and compare KMeans clusters to the ground-truth component labels.


## 0. Knobs

These are ordinary notebook knobs. Change them, rerun downward, and inspect how the geometry changes.


In [ ]:
from __future__ import annotations

# SEED controls the sampled points. GEOMETRY_SEED controls the GMM means and subspace.
SEED = 7
GEOMETRY_SEED = 11
N_TRAIN = 1_500
N_REFERENCE = 1_500
N_CLUSTERS = 8
SIGMA_Z = 0.20
SIGMA_ETA = 1.0e-3

# First sanity check: no projection trickery, just a directly plottable 2D GMM.
TWO_D_DATA_DIM = 2
TWO_D_INTRINSIC_DIM = 2

# Second sanity check: same 2D latent mixture, projected into a larger ambient space.
HIGH_D_DATA_DIM = 256
HIGH_D_INTRINSIC_DIM = 2

# Third sanity check: keep d / D fixed while intrinsic dimension d increases.
FIXED_RATIO_CASES = [
    (2, 32),
    (4, 64),
    (8, 128),
    (16, 256),
]

EMBEDDING_SUBSAMPLE = 700
RUN_TSNE = True
RUN_UMAP = True


## 1. Imports and repo path setup

The notebook lives under `experiments/synthetic/`, so we add the repo root and `src/` explicitly. This keeps the notebook using the local checkout rather than an old installed package.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

repo_root = Path.cwd().resolve()
for candidate in (repo_root, *repo_root.parents):
    if (candidate / "src").exists() and (candidate / "experiments").exists():
        repo_root = candidate
        break
else:
    raise RuntimeError("Could not find the repo root from the current working directory.")

src_root = repo_root / "src"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

# UMAP is useful for visual inspection, but the notebook should still run without it.
try:
    import umap
    HAS_UMAP = True
except ModuleNotFoundError:
    HAS_UMAP = False

from mp_config import clone_config, get_base_config
from mp_data import build_data_bundle

torch.manual_seed(SEED)
np.random.seed(SEED)

print("repo_root:", repo_root)
print("umap available:", HAS_UMAP)


## 2. Build a 2D GMM config

Here `d = D = 2`, so the generated tensor should be directly plottable. The labels are the true Gaussian-component IDs.


In [ ]:
config_2d = clone_config(get_base_config())

# We build the config field by field so the data assumptions are visible.
config_2d.study.name = "synthetic_gmm_data_sanity"
config_2d.outputs.run_name = "gmm_2d_data_sanity"

config_2d.data.dataset = "gmm_subspace"
config_2d.data.n_train = int(N_TRAIN)
config_2d.data.n_eval = int(N_REFERENCE)
config_2d.data.data_dim = int(TWO_D_DATA_DIM)
config_2d.data.intrinsic_dim = int(TWO_D_INTRINSIC_DIM)
config_2d.data.n_clusters = int(N_CLUSTERS)
config_2d.data.sigma_z = float(SIGMA_Z)
config_2d.data.sigma_eta = float(SIGMA_ETA)
config_2d.data.geometry_seed = int(GEOMETRY_SEED)

config_2d.training.seed = int(SEED)

print("2D config:")
print(json.dumps({
    "dataset": config_2d.data.dataset,
    "n_train": config_2d.data.n_train,
    "n_eval": config_2d.data.n_eval,
    "data_dim_D": config_2d.data.data_dim,
    "intrinsic_dim_d": config_2d.data.intrinsic_dim,
    "n_clusters": config_2d.data.n_clusters,
    "sigma_z": config_2d.data.sigma_z,
    "sigma_eta": config_2d.data.sigma_eta,
    "geometry_seed": config_2d.data.geometry_seed,
    "sample_seed": config_2d.training.seed,
}, indent=2))


## 3. Materialize the 2D train/reference tensors

This goes through the canonical loader, then we immediately unpack the tensors and metadata so we can inspect them directly.


In [ ]:
bundle_2d = build_data_bundle(config_2d)

# The train/reference tensors are separate synthetic draws; labels are true component IDs.
train_2d = bundle_2d["train"]
reference_2d = bundle_2d["reference"]
labels_2d = bundle_2d["train_labels"]
reference_labels_2d = bundle_2d["reference_labels"]
basis_2d = bundle_2d["geometry"]["basis"]
means_2d = bundle_2d["geometry"]["means"]

print("train shape      :", tuple(train_2d.shape), train_2d.dtype)
print("reference shape  :", tuple(reference_2d.shape), reference_2d.dtype)
print("basis shape      :", tuple(basis_2d.shape))
print("means shape      :", tuple(means_2d.shape))
print("source ids       :", bundle_2d["train_source_id"], "vs", bundle_2d["reference_source_id"])
print("validation       :", bundle_2d.get("data_validation"))
print("label counts     :", torch.bincount(labels_2d, minlength=N_CLUSTERS).tolist())


## 4. Check the 2D basis and draw the points

For `d = D = 2`, the basis should be a 2x2 orthonormal rotation/reflection. The cluster means are sampled in latent space and then projected to ambient space by `means @ basis.T`.


In [ ]:
gram_2d = basis_2d.T @ basis_2d
identity_2d = torch.eye(config_2d.data.intrinsic_dim)
basis_error_2d = (gram_2d - identity_2d).abs().max().item()
# Ambient means are what the latent GMM centers look like after applying the basis.
ambient_means_2d = means_2d @ basis_2d.T

print("max |B.T B - I|:", basis_error_2d)
print("latent means first rows:")
print(means_2d[:5])
print("ambient means first rows:")
print(ambient_means_2d[:5])

fig, ax = plt.subplots(figsize=(6, 6))
scatter = ax.scatter(
    train_2d[:, 0],
    train_2d[:, 1],
    c=labels_2d,
    s=8,
    alpha=0.55,
    cmap="tab10",
)
ax.scatter(
    ambient_means_2d[:, 0],
    ambient_means_2d[:, 1],
    c=torch.arange(N_CLUSTERS),
    marker="x",
    s=120,
    linewidths=2.5,
    cmap="tab10",
)
ax.set_title("2D GMM samples with true component labels")
ax.set_xlabel("x[0]")
ax.set_ylabel("x[1]")
ax.axis("equal")
ax.grid(alpha=0.25)
fig.colorbar(scatter, ax=ax, label="true cluster")
display(fig)
plt.close(fig)


## 5. Cluster sanity check in 2D

We know the true component labels, so KMeans should roughly recover them when the clusters are separated. ARI and NMI are label-permutation invariant, so they are appropriate here.


In [ ]:
x_2d_np = train_2d.numpy()
labels_2d_np = labels_2d.numpy()

# KMeans does not see the true labels; ARI/NMI compare its discovered clusters afterward.
kmeans_2d = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=SEED)
pred_2d = kmeans_2d.fit_predict(x_2d_np)

ari_2d = adjusted_rand_score(labels_2d_np, pred_2d)
nmi_2d = normalized_mutual_info_score(labels_2d_np, pred_2d)
silhouette_2d = silhouette_score(x_2d_np, pred_2d)

print("2D KMeans ARI       :", ari_2d)
print("2D KMeans NMI       :", nmi_2d)
print("2D silhouette       :", silhouette_2d)
print("predicted counts    :", np.bincount(pred_2d, minlength=N_CLUSTERS).tolist())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(train_2d[:, 0], train_2d[:, 1], c=labels_2d, s=8, alpha=0.55, cmap="tab10")
axes[0].set_title("ground-truth labels")
axes[0].axis("equal")
axes[0].grid(alpha=0.25)
axes[1].scatter(train_2d[:, 0], train_2d[:, 1], c=pred_2d, s=8, alpha=0.55, cmap="tab10")
axes[1].set_title("KMeans labels")
axes[1].axis("equal")
axes[1].grid(alpha=0.25)
fig.tight_layout()
display(fig)
plt.close(fig)


## 6. Build a projected high-dimensional GMM config

Now we keep the intrinsic mixture two-dimensional, but project it into a larger ambient space. The expected behavior is that the true latent projection `x @ B` should recover the same cluster picture, while the raw ambient vectors live in `D` dimensions.


In [ ]:
config_hd = clone_config(get_base_config())

# Same GMM family as above, but now D is much larger than the true intrinsic dimension d.
config_hd.study.name = "synthetic_gmm_data_sanity"
config_hd.outputs.run_name = "gmm_highD_data_sanity"

config_hd.data.dataset = "gmm_subspace"
config_hd.data.n_train = int(N_TRAIN)
config_hd.data.n_eval = int(N_REFERENCE)
config_hd.data.data_dim = int(HIGH_D_DATA_DIM)
config_hd.data.intrinsic_dim = int(HIGH_D_INTRINSIC_DIM)
config_hd.data.n_clusters = int(N_CLUSTERS)
config_hd.data.sigma_z = float(SIGMA_Z)
config_hd.data.sigma_eta = float(SIGMA_ETA)
config_hd.data.geometry_seed = int(GEOMETRY_SEED)

config_hd.training.seed = int(SEED)

print("high-D config:")
print(json.dumps({
    "data_dim_D": config_hd.data.data_dim,
    "intrinsic_dim_d": config_hd.data.intrinsic_dim,
    "d_over_D": config_hd.data.intrinsic_dim / config_hd.data.data_dim,
    "n_clusters": config_hd.data.n_clusters,
    "sigma_z": config_hd.data.sigma_z,
    "sigma_eta": config_hd.data.sigma_eta,
}, indent=2))


## 7. Materialize and inspect the high-D data

The key sanity check is that the data should mostly live in the span of the saved basis `B`, up to the small ambient noise `sigma_eta`.


In [ ]:
bundle_hd = build_data_bundle(config_hd)

train_hd = bundle_hd["train"]
reference_hd = bundle_hd["reference"]
labels_hd = bundle_hd["train_labels"]
basis_hd = bundle_hd["geometry"]["basis"]
means_hd = bundle_hd["geometry"]["means"]

# Since B has orthonormal columns, x @ B recovers coordinates in the planted subspace.
latent_hd = train_hd @ basis_hd
train_hd_projected = latent_hd @ basis_hd.T
residual_hd = train_hd - train_hd_projected

gram_hd = basis_hd.T @ basis_hd
identity_hd = torch.eye(config_hd.data.intrinsic_dim)
basis_error_hd = (gram_hd - identity_hd).abs().max().item()
# This should be tiny when sigma_eta is tiny; otherwise the data is not mostly subspace data.
residual_fraction_hd = (residual_hd.pow(2).sum() / train_hd.pow(2).sum()).item()

print("train shape             :", tuple(train_hd.shape))
print("reference shape         :", tuple(reference_hd.shape))
print("basis shape             :", tuple(basis_hd.shape))
print("max |B.T B - I|         :", basis_error_hd)
print("residual energy fraction:", residual_fraction_hd)
print("label counts            :", torch.bincount(labels_hd, minlength=N_CLUSTERS).tolist())
print("validation              :", bundle_hd.get("data_validation"))


## 8. Look at the high-D data through the true latent coordinates

Because `B` is known, the most honest visualization is `x @ B`. If this does not look like the 2D mixture, the generator is wrong.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
scatter = ax.scatter(
    latent_hd[:, 0],
    latent_hd[:, 1],
    c=labels_hd,
    s=8,
    alpha=0.55,
    cmap="tab10",
)
ax.scatter(
    means_hd[:, 0],
    means_hd[:, 1],
    c=torch.arange(N_CLUSTERS),
    marker="x",
    s=120,
    linewidths=2.5,
    cmap="tab10",
)
ax.set_title(f"High-D GMM projected back to true latent coordinates, d={HIGH_D_INTRINSIC_DIM}, D={HIGH_D_DATA_DIM}")
ax.set_xlabel("x @ B [0]")
ax.set_ylabel("x @ B [1]")
ax.axis("equal")
ax.grid(alpha=0.25)
fig.colorbar(scatter, ax=ax, label="true cluster")
display(fig)
plt.close(fig)


## 9. PCA, t-SNE, and optional UMAP on the high-D data

PCA should find the subspace cleanly when the ambient noise is tiny. t-SNE and UMAP are not proof, but they are useful visual sanity checks: do the true component labels still organize the embedding?


In [ ]:
subsample_count = min(EMBEDDING_SUBSAMPLE, train_hd.shape[0])
subsample_generator = torch.Generator().manual_seed(SEED)
# Embeddings are only for visual sanity checks, so we subsample to keep the notebook snappy.
embedding_indices = torch.randperm(train_hd.shape[0], generator=subsample_generator)[:subsample_count]

x_embed = train_hd[embedding_indices].numpy()
labels_embed = labels_hd[embedding_indices].numpy()

pca_hd = PCA(n_components=min(16, HIGH_D_DATA_DIM), random_state=SEED)
pca_hd_scores = pca_hd.fit_transform(x_embed)
pca_hd_2d = pca_hd_scores[:, :2]
# If the generator is right, the first d PCs should explain almost all variance.
explained_first_d = pca_hd.explained_variance_ratio_[:HIGH_D_INTRINSIC_DIM].sum()

print("embedding subset shape:", x_embed.shape)
print("PCA explained variance first d components:", explained_first_d)
print("PCA explained variance first 8 components:", pca_hd.explained_variance_ratio_[:8])

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(pca_hd_2d[:, 0], pca_hd_2d[:, 1], c=labels_embed, s=10, alpha=0.65, cmap="tab10")
ax.set_title("PCA view of high-D GMM")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(alpha=0.25)
display(fig)
plt.close(fig)


In [ ]:
if RUN_TSNE:
    tsne_hd = TSNE(
        n_components=2,
        perplexity=min(30, max(5, (subsample_count - 1) // 3)),
        init="pca",
        learning_rate="auto",
        random_state=SEED,
    )
    tsne_hd_2d = tsne_hd.fit_transform(x_embed)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(tsne_hd_2d[:, 0], tsne_hd_2d[:, 1], c=labels_embed, s=10, alpha=0.65, cmap="tab10")
    ax.set_title("t-SNE view of high-D GMM")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.grid(alpha=0.25)
    display(fig)
    plt.close(fig)
else:
    print("Skipping t-SNE because RUN_TSNE=False")


In [ ]:
if RUN_UMAP and HAS_UMAP:
    umap_hd = umap.UMAP(
        n_components=2,
        n_neighbors=20,
        min_dist=0.10,
        random_state=SEED,
    )
    umap_hd_2d = umap_hd.fit_transform(x_embed)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(umap_hd_2d[:, 0], umap_hd_2d[:, 1], c=labels_embed, s=10, alpha=0.65, cmap="tab10")
    ax.set_title("UMAP view of high-D GMM")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.grid(alpha=0.25)
    display(fig)
    plt.close(fig)
elif RUN_UMAP and not HAS_UMAP:
    print("Skipping UMAP because the `umap-learn` package is not installed.")
else:
    print("Skipping UMAP because RUN_UMAP=False")


## 10. Cluster sanity check in ambient vs true latent coordinates

If the subspace generator is behaving, clustering in true latent coordinates should be at least as clean as clustering in the full ambient vectors. With tiny ambient noise, they should usually be similar.


In [ ]:
kmeans_hd_ambient = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=SEED)
pred_hd_ambient = kmeans_hd_ambient.fit_predict(train_hd.numpy())

# Latent clustering uses the planted coordinates; ambient clustering uses the full D-vector.
kmeans_hd_latent = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=SEED)
pred_hd_latent = kmeans_hd_latent.fit_predict(latent_hd.numpy())

ari_hd_ambient = adjusted_rand_score(labels_hd.numpy(), pred_hd_ambient)
ari_hd_latent = adjusted_rand_score(labels_hd.numpy(), pred_hd_latent)
nmi_hd_ambient = normalized_mutual_info_score(labels_hd.numpy(), pred_hd_ambient)
nmi_hd_latent = normalized_mutual_info_score(labels_hd.numpy(), pred_hd_latent)

print("ambient KMeans ARI:", ari_hd_ambient)
print("latent  KMeans ARI:", ari_hd_latent)
print("ambient KMeans NMI:", nmi_hd_ambient)
print("latent  KMeans NMI:", nmi_hd_latent)


## 11. Fixed `d / D` sweep while `d` goes up

Now we check a family where the intrinsic-to-ambient ratio is fixed. The point is not to train models here. The point is to confirm that the data geometry stays controlled as both dimensions increase.


In [ ]:
ratio_records = []

for intrinsic_dim, data_dim in FIXED_RATIO_CASES:
    # Build each case explicitly so every row in the table has a known data geometry.
    config_ratio = clone_config(get_base_config())
    config_ratio.study.name = "synthetic_gmm_data_sanity"
    config_ratio.outputs.run_name = f"gmm_ratio_d{intrinsic_dim}_D{data_dim}"
    config_ratio.data.dataset = "gmm_subspace"
    config_ratio.data.n_train = int(N_TRAIN)
    config_ratio.data.n_eval = int(N_REFERENCE)
    config_ratio.data.data_dim = int(data_dim)
    config_ratio.data.intrinsic_dim = int(intrinsic_dim)
    config_ratio.data.n_clusters = int(N_CLUSTERS)
    config_ratio.data.sigma_z = float(SIGMA_Z)
    config_ratio.data.sigma_eta = float(SIGMA_ETA)
    config_ratio.data.geometry_seed = int(GEOMETRY_SEED)
    config_ratio.training.seed = int(SEED)

    bundle_ratio = build_data_bundle(config_ratio)
    x_ratio = bundle_ratio["train"]
    y_ratio = bundle_ratio["train_labels"]
    basis_ratio = bundle_ratio["geometry"]["basis"]
    # Project to the planted subspace, then reconstruct to measure off-subspace energy.
    latent_ratio = x_ratio @ basis_ratio
    x_ratio_projected = latent_ratio @ basis_ratio.T
    residual_ratio = x_ratio - x_ratio_projected

    gram_ratio = basis_ratio.T @ basis_ratio
    identity_ratio = torch.eye(intrinsic_dim)
    basis_error_ratio = (gram_ratio - identity_ratio).abs().max().item()
    residual_fraction_ratio = (residual_ratio.pow(2).sum() / x_ratio.pow(2).sum()).item()

    pca_components = min(max(intrinsic_dim, 2), data_dim, 32)
    pca_ratio = PCA(n_components=pca_components, random_state=SEED)
    pca_ratio.fit(x_ratio.numpy())
    # This is a simple check that PCA sees the planted d-dimensional structure.
    pca_first_d_variance = pca_ratio.explained_variance_ratio_[:intrinsic_dim].sum()

    # Compare clustering in the true latent coordinates against clustering in ambient space.
    pred_ratio_latent = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=SEED).fit_predict(latent_ratio.numpy())
    pred_ratio_ambient = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=SEED).fit_predict(x_ratio.numpy())

    ratio_records.append({
        "d": int(intrinsic_dim),
        "D": int(data_dim),
        "d_over_D": float(intrinsic_dim / data_dim),
        "train_shape": tuple(x_ratio.shape),
        "basis_error": basis_error_ratio,
        "residual_energy_fraction": residual_fraction_ratio,
        "pca_first_d_variance": float(pca_first_d_variance),
        "ari_latent": adjusted_rand_score(y_ratio.numpy(), pred_ratio_latent),
        "ari_ambient": adjusted_rand_score(y_ratio.numpy(), pred_ratio_ambient),
        "nmi_latent": normalized_mutual_info_score(y_ratio.numpy(), pred_ratio_latent),
        "nmi_ambient": normalized_mutual_info_score(y_ratio.numpy(), pred_ratio_ambient),
    })

ratio_df = pd.DataFrame(ratio_records)
display(ratio_df)


## 12. Plot the fixed-ratio diagnostics

These plots should be boring in a good way: orthonormality errors near zero, residual energy near the configured ambient noise level, high PCA variance in the first `d` components, and stable clustering metrics.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(ratio_df["d"], ratio_df["residual_energy_fraction"], marker="o")
axes[0].set_title("off-subspace residual energy")
axes[0].set_xlabel("intrinsic dimension d")
axes[0].set_ylabel("residual / total energy")
axes[0].grid(alpha=0.25)

axes[1].plot(ratio_df["d"], ratio_df["pca_first_d_variance"], marker="o")
axes[1].set_title("PCA variance captured by first d PCs")
axes[1].set_xlabel("intrinsic dimension d")
axes[1].set_ylabel("explained variance")
axes[1].set_ylim(0.0, 1.05)
axes[1].grid(alpha=0.25)

axes[2].plot(ratio_df["d"], ratio_df["ari_latent"], marker="o", label="latent")
axes[2].plot(ratio_df["d"], ratio_df["ari_ambient"], marker="s", label="ambient")
axes[2].set_title("KMeans ARI vs true components")
axes[2].set_xlabel("intrinsic dimension d")
axes[2].set_ylabel("ARI")
axes[2].set_ylim(-0.05, 1.05)
axes[2].legend()
axes[2].grid(alpha=0.25)

fig.tight_layout()
display(fig)
plt.close(fig)


## 13. Inspect the largest fixed-ratio case in 2D embeddings

We now take the last `(d, D)` pair from the sweep and visualize it. This is the most useful quick sanity check before deciding whether the high-dimensional geometry is behaving as expected.


In [ ]:
largest_d, largest_D = FIXED_RATIO_CASES[-1]

config_largest = clone_config(get_base_config())
config_largest.study.name = "synthetic_gmm_data_sanity"
config_largest.outputs.run_name = f"gmm_largest_ratio_d{largest_d}_D{largest_D}"
config_largest.data.dataset = "gmm_subspace"
config_largest.data.n_train = int(N_TRAIN)
config_largest.data.n_eval = int(N_REFERENCE)
config_largest.data.data_dim = int(largest_D)
config_largest.data.intrinsic_dim = int(largest_d)
config_largest.data.n_clusters = int(N_CLUSTERS)
config_largest.data.sigma_z = float(SIGMA_Z)
config_largest.data.sigma_eta = float(SIGMA_ETA)
config_largest.data.geometry_seed = int(GEOMETRY_SEED)
config_largest.training.seed = int(SEED)

bundle_largest = build_data_bundle(config_largest)
x_largest = bundle_largest["train"]
y_largest = bundle_largest["train_labels"]
basis_largest = bundle_largest["geometry"]["basis"]
latent_largest = x_largest @ basis_largest

# We inspect one representative high-dimensional case more closely in 2D projections.
subsample_count_largest = min(EMBEDDING_SUBSAMPLE, x_largest.shape[0])
embedding_indices_largest = torch.randperm(x_largest.shape[0], generator=torch.Generator().manual_seed(SEED))[:subsample_count_largest]
x_largest_embed = x_largest[embedding_indices_largest].numpy()
latent_largest_embed = latent_largest[embedding_indices_largest].numpy()
y_largest_embed = y_largest[embedding_indices_largest].numpy()

print("largest case:", {"d": largest_d, "D": largest_D, "shape": tuple(x_largest.shape)})
print("latent embed shape:", latent_largest_embed.shape)


In [ ]:
pca_largest = PCA(n_components=2, random_state=SEED)
pca_largest_2d = pca_largest.fit_transform(x_largest_embed)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(latent_largest_embed[:, 0], latent_largest_embed[:, 1], c=y_largest_embed, s=10, alpha=0.65, cmap="tab10")
axes[0].set_title("first two true latent coordinates")
axes[0].set_xlabel("latent 1")
axes[0].set_ylabel("latent 2")
axes[0].grid(alpha=0.25)

axes[1].scatter(pca_largest_2d[:, 0], pca_largest_2d[:, 1], c=y_largest_embed, s=10, alpha=0.65, cmap="tab10")
axes[1].set_title("PCA view")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].grid(alpha=0.25)

fig.tight_layout()
display(fig)
plt.close(fig)


In [ ]:
if RUN_TSNE:
    tsne_largest = TSNE(
        n_components=2,
        perplexity=min(30, max(5, (subsample_count_largest - 1) // 3)),
        init="pca",
        learning_rate="auto",
        random_state=SEED,
    )
    tsne_largest_2d = tsne_largest.fit_transform(x_largest_embed)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(tsne_largest_2d[:, 0], tsne_largest_2d[:, 1], c=y_largest_embed, s=10, alpha=0.65, cmap="tab10")
    ax.set_title(f"t-SNE view, d={largest_d}, D={largest_D}")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.grid(alpha=0.25)
    display(fig)
    plt.close(fig)
else:
    print("Skipping t-SNE because RUN_TSNE=False")


In [ ]:
if RUN_UMAP and HAS_UMAP:
    umap_largest = umap.UMAP(
        n_components=2,
        n_neighbors=20,
        min_dist=0.10,
        random_state=SEED,
    )
    umap_largest_2d = umap_largest.fit_transform(x_largest_embed)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(umap_largest_2d[:, 0], umap_largest_2d[:, 1], c=y_largest_embed, s=10, alpha=0.65, cmap="tab10")
    ax.set_title(f"UMAP view, d={largest_d}, D={largest_D}")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.grid(alpha=0.25)
    display(fig)
    plt.close(fig)
elif RUN_UMAP and not HAS_UMAP:
    print("Skipping UMAP because the `umap-learn` package is not installed.")
else:
    print("Skipping UMAP because RUN_UMAP=False")


## 14. Summary checklist

Before using this data in a training sweep, the checks we want are:

- train/reference source IDs are distinct
- `B.T @ B` is close to identity
- off-subspace residual energy is tiny when `sigma_eta` is tiny
- in true latent coordinates, labels form the expected GMM clusters
- PCA/t-SNE/UMAP views do not contradict the expected geometry
- KMeans has sensible ARI/NMI against ground-truth cluster labels
